# A1.13 · Resource overload

**Function A — Securing AI Architectures → CyberTravels' Architecture, and Every Risk It Carries**  ·  *Security of AI*

Builds on **[A1.12 · Cascading hallucination](https://spbreed.github.io/cyber-commons/lessons/A1.12.html)**.

| | |
|---|---|
| Tools used | OpenTelemetry |

## What this lesson is

**What it covers.** Run a loop with no ceiling and count what it consumes before anything notices.

**Why a security engineer needs it.** An agent consumes budget, tokens, API quota or downstream capacity without bound, and the failure is denial of service against your own systems. The control it builds is: budgets and stop conditions bound to the loop (A3.4).

This is a **risk** lesson: it shows the failure happening before anything tries to stop it, so the control that follows is answering something you have already watched go wrong.

## 1 · The hook

The loop had no ceiling, so it ran until something outside it stopped the run. That something was the invoice. In a different configuration it is a rate limit on a system you do not own, which is somebody else's outage.

> **At CyberTravels.** A booking loop with no ceiling runs until something outside it stops the run. At CyberTravels that something is either the travel API's rate limit — somebody else's outage — or the invoice.

## 2 · The framework

```
   plan --> act --> observe --> plan --> act --> observe --> ...
     ^                                                        |
     +--------------------------------------------------------+

   no token ceiling . no step ceiling . no wall clock . no spend cap
   -> the stop condition is external: an invoice, a rate limit, a person
```

**OWASP T4 — Resource Overload. LLM10 — Unbounded Consumption.**

The **agent_runtime** loops: plan, act, observe, decide again. The loop is the
component that makes an agent an agent, and a loop with no exit condition runs
until something outside it intervenes.

What intervenes, in practice, is a bill, a rate limit, or a person at 3am.

Four resources drain, and they fail differently:

**Tokens and money** — the visible one, discovered on an invoice.

**Downstream capacity** — the one that hurts other people. An agent retrying a
failing API in a tight loop is a denial-of-service attack on your own service,
launched from inside your perimeter by something with valid credentials.

**Rate limit budget** — shared with the humans who need it. The agent exhausts
the quota and the on-call engineer cannot query the API they need.

**Wall-clock time in a critical path** — a workflow step that never returns.

This is a security risk rather than a cost problem for two reasons. It is
**reachable by an attacker**: a task that cannot succeed is easy to construct
via A1.3, and costs the attacker nothing. And it is **availability**, which is
one third of the triad regardless of how the outage was caused.

> **Where this lands on the reference architecture.**
>
> ```
> ingress -> orchestrator -> agent_runtime -> model
>                                |              |
>                          messaging        tools / mcp
>                                |              |
>                       knowledge / memory   egress
>            identity + policy wrap every call · observability records it
> ```

## 3 · The risk, realised

A task that cannot succeed, and a loop with no ceiling.

## 4 · The check, as a skill

CyberTravels' agent does not know the task is impossible. The skill gives it one, measures the three costs, and reports the one that lands on somebody else: the downstream capacity its retries consumed.

### The skill — [`skills/threats/unbounded-loop-cost-probe/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/threats/unbounded-loop-cost-probe/SKILL.md)

```yaml
name: unbounded-loop-cost-probe
description: >-
  Give an agent a task it cannot complete and measure what it spends and who
  else pays — tokens, wall time, and the downstream capacity consumed by its
  retries. Use when reviewing loop termination, retry policy, budgets, or a
  scheduled agent nobody watches.
allowed-tools: Read, Grep, Glob
```

# The agent does not know the task is impossible

Resource overload rarely needs an attacker. It needs a task with no completion
condition and a loop with no stop condition, and the cost lands in two places:
your bill, and a downstream service's capacity — where the rejections hit
whoever else was using it.

## When to use this

Before running any agent unattended or on a schedule, and after adding a retry.

## Procedure

**1 — Find the stop conditions.** Step ceiling, token budget, wall-clock
deadline, cost ceiling, and a condition that recognises "this cannot be done".
Record which exist. A loop whose only exit is success has no exit.

**2 — Construct an impossible-but-plausible task.** Not malformed — plausible.
A query against data that does not exist, a fix for a test that cannot pass. The
agent must believe it is making progress.

**3 — Run it with instrumentation and a hard external kill.** The kill is the
safety net; if it is the thing that stops the run, that is the result.

**4 — Record the three costs.** Tokens and money; wall-clock; and downstream
calls — with the rejection rate the downstream started returning. The third is
the one that turns your incident into somebody else's.

**5 — Set the budget from the measurement.** A ceiling chosen from an observed
distribution is defensible; one chosen from a round number is a guess. State
what a legitimate run costs at p95 and set the ceiling above that.

## Output contract

```json
{
  "stop_conditions": {"steps": false, "tokens": false, "wallclock": false, "cost": false, "impossibility": false},
  "run": {"stopped_by": "condition|external_kill", "steps": 0, "tokens": 0, "seconds": 0},
  "downstream": {"calls": 0, "rejections": 0, "affected_others": true},
  "recommended_budget": {"basis": "p95 of legitimate runs", "steps": 0, "tokens": 0}
}
```

## Failure modes

- **Using a malformed task.** The agent gives up, and you learn nothing.
- **Counting only tokens.** The downstream capacity is the externality.
- **Setting a round-number ceiling.** Measure first, or the budget either
  breaks legitimate runs or never fires.

In [ ]:
# The code is not in this notebook. It is the file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/threats/unbounded-loop-cost-probe/scripts/unbounded_loop_cost_probe.py
SCRIPT = "skills/threats/unbounded-loop-cost-probe/scripts/unbounded_loop_cost_probe.py"

import glob, os, subprocess, sys

# The skills tree: the attached dataset on Kaggle, the checkout locally.
_ROOTS = sorted(glob.glob("/kaggle/input/**/cyber-commons-skills", recursive=True)) + [".", "..", "../.."]
_root = next((r for r in _ROOTS if os.path.isfile(os.path.join(r, SCRIPT))), None)
if _root is None:
    raise SystemExit("skills tree not found. On Kaggle add the dataset "
                     "cybercommons/cyber-commons-skills; locally run from a checkout.")

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

An agent given an impossible task loops until the notebook's own safety net stops it, spending hundreds of thousands of tokens and exhausting a downstream service's capacity — with the rejections landing on whoever else was using that service.

## Your turn

Find the ceiling on one agent loop you run. If there is a token budget but no cap on downstream calls, the cost is bounded and the availability risk is not.

---

**Next → [A1.14 · Repudiation and untraceability](https://spbreed.github.io/cyber-commons/lessons/A1.14.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.13.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.13.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*